# A Fourier layer, step by step

Section 7.3 of the notes.

A convolution multiplies each wave of a periodic function by a number. This
notebook shows it on a random function, and then shows what a pointwise
activation does: it creates the fast waves that the Fourier part drops.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
plt.rcParams.update(
    {
        "font.family": "STIXGeneral",
        "mathtext.fontset": "stix",
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

In [ ]:
def random_function(x, n_waves, rng):
    # A random sum of the first n_waves waves with decaying coefficients.
    k = np.arange(1, n_waves + 1)
    c = 1 / (1 + k**2 / 16)
    a, b = rng.standard_normal((2, n_waves)) * c
    return (
        a @ np.cos(np.pi * np.outer(k, x)) + b @ np.sin(np.pi * np.outer(k, x))
    ) / np.sqrt((c**2).sum())


def coefficients(v):
    # Fourier coefficients of grid values on (-1, 1), for k = 0, 1, ..., d/2.
    return np.fft.rfft(v) / v.size


d = 256
x = -1 + 2 * np.arange(d) / d
v = random_function(x, 16, np.random.default_rng(4))
k = np.arange(d // 2 + 1)
kappa_hat2 = np.exp(
    -((k / 5) ** 2)
)  # twice the kernel coefficients: a smoothing convolution
kappa = np.roll(
    np.fft.irfft(kappa_hat2 / 2 * d, n=d), d // 2
)  # the kernel at z = x, centered at 0
v_hat = coefficients(v)
w = np.fft.irfft(kappa_hat2 * v_hat * d, n=d)

In [ ]:
from matplotlib.patches import ConnectionPatch

fig, axes = plt.subplots(
    2, 3, figsize=(5.0, 2.9), gridspec_kw=dict(wspace=1.05, hspace=1.35)
)
kk = k[:21]
blue, orange, gray = "#2a78d6", "#eb6834", "#555555"
ax = axes[0, 0]
ax.plot(x, v, color=blue, lw=1.1)
ax.set(xlabel="$x$", title="input $v(x)$")
ax = axes[0, 1]
ax.stem(kk, np.abs(v_hat[:21]), linefmt=blue, markerfmt="o", basefmt=" ")
ax.set(xlabel="wave $k$", title=r"its coefficients $|\hat v_k|$")
ax = axes[0, 2]
ax.stem(
    kk, np.abs((kappa_hat2 * v_hat)[:21]), linefmt=orange, markerfmt="o", basefmt=" "
)
ax.set(
    xlabel="wave $k$",
    title=r"new coefficients $|2\hat\varkappa_k\hat v_k|$",
    ylim=axes[0, 1].get_ylim(),
)
ax = axes[1, 0]
ax.plot(x, kappa, color="black", lw=1.1)
ax.set(xlabel="$z$", title=r"kernel $\varkappa(z)$")
ax = axes[1, 1]
ax.plot(kk, kappa_hat2[:21], color="black", lw=1.1, marker="o", ms=2)
ax.set(
    xlabel="wave $k$", title=r"its coefficients $2\hat\varkappa_k$", ylim=(-0.05, 1.1)
)
ax = axes[1, 2]
ax.plot(x, v, color=blue, lw=0.8, alpha=0.35)
ax.plot(x, w, color=orange, lw=1.2)
ax.set(xlabel="$x$", title=r"output $(\mathcal{C}v)(x)$")
for line in axes[0, 1].lines + axes[0, 2].lines:
    line.set_markersize(2)


def arrow(ax_a, xy_a, ax_b, xy_b, label, offset):
    # An arrow between two panels, in axes fractions, with a label at its middle.
    fig.add_artist(
        ConnectionPatch(
            xy_a,
            xy_b,
            "axes fraction",
            "axes fraction",
            axesA=ax_a,
            axesB=ax_b,
            arrowstyle="-|>",
            color=gray,
            lw=0.8,
            mutation_scale=7,
        )
    )
    a = fig.transFigure.inverted().transform(ax_a.transAxes.transform(xy_a))
    b = fig.transFigure.inverted().transform(ax_b.transAxes.transform(xy_b))
    fig.text(
        *((a + b) / 2 + offset), label, color=gray, fontsize=7, ha="center", va="center"
    )


arrow(axes[0, 0], (1.05, 0.5), axes[0, 1], (-0.33, 0.5), "FFT", (0, 0.035))
arrow(axes[1, 0], (1.05, 0.5), axes[1, 1], (-0.33, 0.5), "FFT", (0, 0.035))
arrow(axes[0, 1], (1.05, 0.5), axes[0, 2], (-0.33, 0.5), r"$\times$", (0, 0.035))
arrow(axes[1, 1], (1.05, 1.0), axes[0, 2], (-0.33, 0.35), "", (0, 0))
arrow(axes[0, 2], (0.5, -0.55), axes[1, 2], (0.5, 1.3), "inverse FFT", (0.085, 0))
fig.savefig(FIGURES / "fourier_filter.pdf", bbox_inches="tight")
plt.show()

## 2. The activation creates fast waves

Keep only the waves $k\le4$ of the input, then apply $\tanh(5\,\cdot)$ at each
point: the result has steep interfaces, and its coefficients reach far beyond
$k = 4$.

In [ ]:
slow_hat = v_hat.copy()
slow_hat[5:] = 0
slow = np.fft.irfft(slow_hat * d, n=d)
sharp = np.tanh(5 * slow)
fig, axes = plt.subplots(1, 2, figsize=(3.4, 0.95), gridspec_kw=dict(wspace=0.45))
axes[0].plot(x, slow, color="#2a78d6", lw=1.1, label=r"$v$: waves $k\leq 4$")
axes[0].plot(x, sharp, color="#eb6834", lw=1.2, label=r"$\tanh(5v)$")
axes[0].set(xlabel="$x$")
axes[1].semilogy(k[:61], np.abs(coefficients(sharp))[:61], "o", ms=2, color="#eb6834")
axes[1].semilogy(k[:5], np.abs(coefficients(slow))[:5], "s", ms=3, color="#2a78d6")
axes[1].set(xlabel="wave $k$", title="sizes of the coefficients", ylim=(1e-6, 2))
axes[0].legend(
    frameon=False,
    fontsize=6,
    loc="lower left",
    bbox_to_anchor=(0.0, 0.98),
    ncol=2,
    handlelength=1.2,
    columnspacing=0.8,
    borderaxespad=0,
)
fig.savefig(FIGURES / "activation_waves.pdf", bbox_inches="tight")
plt.show()